# 01 — Dados
Este caderno lê as 19 bases brutas da Economática e do BCB e as transforma em uma base mensal limpa.

In [ ]:
from pathlib import Path
import pandas as pd

PASTA_RAW = Path("../data/raw")

arquivos = sorted(PASTA_RAW.glob("*"))
arquivos = [a for a in arquivos if a.name != ".gitkeep"]

print(len(arquivos), "arquivos em data/raw")
for a in arquivos:
    print(a.name)

## 1. Como a Economática entrega os dados
Abrimos uma base sem nenhum tratamento, para ver o formato original.

In [ ]:
caminho = PASTA_RAW / "09_divida_liquida.xlsx"
bruta = pd.read_excel(caminho, header=None, engine="openpyxl")

print("formato (linhas, colunas):", bruta.shape)
print()
print(bruta.iloc[:6, :3].to_string())

In [ ]:
nome_da_aba = pd.ExcelFile(caminho, engine="openpyxl").sheet_names[0].strip()
cabecalho = bruta.iloc[3].tolist()
codigos = [nome_da_aba] + [str(item).strip().split("\n")[-1].strip() for item in cabecalho[2:]]

print(nome_da_aba)
print(codigos[:5])
print(len(codigos))

In [ ]:
corpo = bruta.iloc[4:].reset_index(drop=True)
datas_brutas = corpo.iloc[:, 0]
valores = corpo.iloc[:, 1:].copy()
valores.columns = codigos
valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")

print(datas_brutas.head(3).tolist(), "...", datas_brutas.tail(2).tolist())
print(valores.shape)
print("células com valor:", int(valores.notna().sum().sum()), "de", valores.size)

## 2. Datas
A Economática entrega os balanços como "1T2017". Convertemos para o último dia do trimestre (31/03/2017), que é a data a que o balanço se refere.

In [ ]:
import re

def converter_trimestre(texto):
    trimestre, ano = re.fullmatch(r"([1-4])T(\d{4})", texto.strip()).groups()
    return pd.Period(year=int(ano), quarter=int(trimestre), freq="Q").end_time.normalize()

datas = [converter_trimestre(t) for t in datas_brutas]
valores.index = pd.DatetimeIndex(datas, name="data")

print(valores.index[:3].tolist())
print(valores.index[-2:].tolist())

## 3. Cadastro das ações
O cadastro diz, para cada código, se a ação está ativa ou cancelada, o setor e o segmento. Vamos conferir se ele cobre as ações da planilha.


In [ ]:
cadastro = pd.read_excel(PASTA_RAW / "01_cadastro_universo.xlsx", header=None, engine="openpyxl", skiprows=4)
cadastro = cadastro.iloc[:, :14]
cadastro.columns = [
    "numero", "nome", "classe", "tipo_de_ativo", "bolsa", "situacao", "pais_sede",
    "nome_repetido", "codigo", "isin", "data_inicio_precos", "subsetor", "setor_economico", "segmento",
]

indice_codigos = pd.Index(codigos)
print("códigos na planilha:", len(indice_codigos), "| únicos:", indice_codigos.nunique())
print("repetidos:", indice_codigos[indice_codigos.duplicated()].tolist())
print("ações no cadastro:", cadastro["codigo"].nunique())
print("na planilha e fora do cadastro:", len(set(indice_codigos) - set(cadastro["codigo"].astype(str).str.strip())))
print(cadastro["situacao"].value_counts().to_dict())

## 4. Lendo todas as bases
Juntamos os passos das seções 1 e 2 em uma função e a aplicamos às 12 bases em formato de matriz (data nas linhas, ação nas colunas).

In [ ]:
MESES = {"Jan": 1, "Fev": 2, "Mar": 3, "Abr": 4, "Mai": 5, "Jun": 6,
         "Jul": 7, "Ago": 8, "Set": 9, "Out": 10, "Nov": 11, "Dez": 12}

def converter_data(valor):
    if hasattr(valor, "year"):
        return pd.Timestamp(valor)
    texto = str(valor).strip()
    if "T" in texto:
        trimestre, ano = texto.split("T")
        return pd.Timestamp(int(ano), int(trimestre) * 3, 1) + pd.offsets.MonthEnd(0)
    mes, ano = texto.split("-")
    return pd.Timestamp(int(ano), MESES[mes], 1) + pd.offsets.MonthEnd(0)

In [ ]:
def ler_matriz(caminho):
    bruta = pd.read_excel(caminho, header=None, engine="openpyxl")
    aba = pd.ExcelFile(caminho, engine="openpyxl").sheet_names[0].strip()

    codigos = [aba] + [str(c).strip().split("\n")[-1].strip() for c in bruta.iloc[3, 2:]]
    datas = [converter_data(d) for d in bruta.iloc[4:, 0]]

    valores = bruta.iloc[4:, 1:].copy()
    valores.columns = codigos
    valores.index = pd.DatetimeIndex(datas, name="data")
    valores = valores.replace("-", float("nan")).apply(pd.to_numeric, errors="coerce")

    return valores.loc[:, ~valores.columns.duplicated()]

In [ ]:
ARQUIVOS = {
    "preco_ajustado": "02_precos_fechamento_ajustados.xlsx",
    "preco_sem_ajuste": "03_precos_sem_ajuste.xlsx",
    "volume": "04_Volume_Diario.xlsx",
    "negocios": "05_Numeros_Negocios.xlsx",
    "valor_mercado": "06_Valor_Mercado.xlsx",
    "acoes": "07_Quantidade_Acoes.xlsx",
    "patrimonio_liquido": "08_Patrimonio_Liquido.xlsx",
    "divida_liquida": "09_divida_liquida.xlsx",
    "ebitda": "010_EBITDA.xlsx",
    "depreciacao": "011_D&A.xlsx",
    "fluxo_caixa_livre": "012_FCFF.xlsx",
    "despesa_financeira": "013_Despesas_Financeiras.xlsx",
}

matrizes = {}
for nome, arquivo in ARQUIVOS.items():
    matrizes[nome] = ler_matriz(PASTA_RAW / arquivo)
    m = matrizes[nome]
    print(f"{nome:20s} {m.shape}  {m.index[0].date()} a {m.index[-1].date()}")

## 5. Ibovespa e taxa livre de risco

O Ibovespa entra só como benchmark. O CDI e a Selic servem de taxa livre de risco, para o fator de mercado e para o Sharpe. Primeiro olhamos o formato de cada arquivo, depois transformamos em séries mensais.

In [ ]:
bruto = pd.read_excel(PASTA_RAW / "14_ibov.xlsx", header=None)

print(bruto.shape)
print(bruto.head(10))
print(bruto.tail(5))

In [ ]:
ibov = bruto.iloc[4:].copy()
ibov.columns = ["data", "fechamento"]
ibov["data"] = pd.to_datetime(ibov["data"])
ibov["fechamento"] = pd.to_numeric(ibov["fechamento"], errors="coerce")
ibov = ibov.set_index("data")["fechamento"]

print(ibov.shape)
print(ibov.isna().sum(), "dias sem valor")
print(ibov.dropna().index.min().date(), "a", ibov.dropna().index.max().date())

In [ ]:
ibov_mensal = ibov.dropna().resample("ME").last()
ibov_ret = ibov_mensal.pct_change()

print(ibov_mensal.head(3))
print(ibov_ret.head(3))
print(ibov_mensal.shape)

In [ ]:
with open(PASTA_RAW / "15_cdi_diario.csv", encoding="utf-8") as f:
    linhas = [f.readline() for _ in range(6)]

for l in linhas:
    print(repr(l))

In [ ]:
cdi = pd.read_csv(PASTA_RAW / "15_cdi_diario.csv", sep=";", decimal=",")
cdi["data"] = pd.to_datetime(cdi["data"], format="%d/%m/%Y")
cdi = cdi.set_index("data")["valor"]

print(cdi.shape)
print(cdi.dtype)
print(cdi.isna().sum(), "sem valor")
print(cdi.index.min().date(), "a", cdi.index.max().date())

In [ ]:
cdi_mensal = (1 + cdi / 100).resample("ME").prod() - 1

print(cdi_mensal.head(3))
print(cdi_mensal.tail(3))
print(cdi_mensal.shape)

In [ ]:
for nome in ["16_selic_meta.csv", "17_selic_apurada.csv"]:
    with open(PASTA_RAW / nome, encoding="utf-8") as f:
        linhas = [f.readline() for _ in range(5)]
    print(nome)
    for l in linhas:
        print(repr(l))
    print()

In [ ]:
selic_meta = pd.read_csv(PASTA_RAW / "16_selic_meta.csv", sep=";", decimal=",", skiprows=1, skipfooter=1, engine="python", names=["data", "meta"])
selic_meta["data"] = pd.to_datetime(selic_meta["data"], format="%d/%m/%Y")
selic_meta = selic_meta.set_index("data")["meta"]

print(selic_meta.shape)
print(selic_meta.dtype)
print(selic_meta.isna().sum(), "sem valor")
print(selic_meta.index.min().date(), "a", selic_meta.index.max().date())
print(selic_meta[selic_meta.diff() != 0].to_string())

In [ ]:
apurada = pd.read_csv(PASTA_RAW / "17_selic_apurada.csv", sep=";", decimal=",", skiprows=1, encoding="utf-8-sig", usecols=["Data", "Fator diário"])
apurada["Data"] = pd.to_datetime(apurada["Data"], format="%d/%m/%Y")
fator = apurada.set_index("Data")["Fator diário"].sort_index()

apurada_mensal = fator.resample("ME").prod() - 1
comparacao = pd.concat([cdi_mensal, apurada_mensal], axis=1, keys=["cdi", "apurada"]).loc["2017-04-30":]

print(fator.shape)
print(fator.index.min().date(), "a", fator.index.max().date())
print((comparacao["cdi"] - comparacao["apurada"]).abs().max())

## 6. Painel mensal por ação

Até aqui lemos os dados como vieram, em frequência diária, mensal e trimestral. O backtest trabalha em base mensal, então o primeiro passo é transformar os preços diários em fechamento e retorno mensal por ação. O fechamento é o último preço disponível de cada ação no mês, e o retorno é o fechamento dividido pelo do mês anterior, menos 1.

In [ ]:
precos = matrizes["preco_ajustado"]

preco_mensal = precos.resample("ME").last()
retorno_mensal = preco_mensal / preco_mensal.shift(1) - 1

print(preco_mensal.shape)
print(preco_mensal.index.min().date(), "a", preco_mensal.index.max().date())
print(preco_mensal.notna().sum(axis=1).head(3))
print(preco_mensal.notna().sum(axis=1).tail(3))

In [ ]:
r = retorno_mensal.stack()

print(r.describe())
print(r.nsmallest(5))
print(r.nlargest(5))

In [ ]:
print((r == 0).mean())
print((r > 1).sum(), "retornos acima de +100%")

janela = slice("2019-02-20", "2019-04-05")
exame = pd.concat(
    [matrizes[k]["GSHP3"].loc[janela] for k in ["preco_ajustado", "preco_sem_ajuste", "volume"]],
    axis=1, keys=["ajustado", "sem_ajuste", "volume"],
).dropna(how="all")
print(exame.to_string())

print(matrizes["preco_ajustado"]["PDGR3"].loc["2025-09-25":"2025-10-31"].dropna().to_string())

In [ ]:
sem_ajuste = matrizes["preco_sem_ajuste"].resample("ME").last()
r_sem = sem_ajuste / sem_ajuste.shift(1) - 1

extremos = r[r > 1].rename("ajustado").to_frame()
extremos["sem_ajuste"] = [r_sem.loc[d, c] for d, c in extremos.index]

print(extremos.sort_values("ajustado", ascending=False).head(15).to_string())
print((extremos["sem_ajuste"] < 0).sum(), "casos com ajustado acima de +100% e sem ajuste negativo")

pdgr = pd.concat(
    [matrizes[k]["PDGR3"].loc["2025-09-30":"2025-10-15"] for k in ["preco_sem_ajuste", "volume"]],
    axis=1, keys=["sem_ajuste", "volume"],
).dropna(how="all")
print(pdgr.to_string())

In [ ]:
print(matrizes["volume"]["PETR4"].dropna().tail(5))
print(matrizes["preco_sem_ajuste"]["PETR4"].dropna().tail(3))

Unidade do volume: o volume do PETR4 fica entre 520 mil e 1,1 milhão por dia, ou seja, R$ mil (cerca de R$ 0,5 a 1,1 bilhão por dia). O piso de liquidez usa R$ mil.

Se uma ação ficou mais de 10 dias corridos sem negociar antes do fim do mês, o preço do mês é descartado. Isso evita carregar preço velho de ação parada.

In [ ]:
def fechamento_mensal(preco_diario, dias_maximos=10):
    ultima_data_com_preco = preco_diario.apply(lambda coluna: coluna.index.to_series().where(coluna.notna()))
    ultima_data_do_mes = ultima_data_com_preco.groupby(preco_diario.index.to_period("M")).max()
    fechamento = preco_diario.groupby(preco_diario.index.to_period("M")).last()
    fim_do_mes = fechamento.index.to_timestamp(how="end").normalize()
    atraso = ultima_data_do_mes.rsub(fim_do_mes, axis=0)
    dias_sem_negociar = atraso.apply(lambda coluna: coluna.dt.days)
    fechamento = fechamento.where(dias_sem_negociar <= dias_maximos)
    fechamento.index = fim_do_mes
    return fechamento

In [ ]:
preco_ajustado_mensal = fechamento_mensal(matrizes["preco_ajustado"])
preco_sem_ajuste_mensal = fechamento_mensal(matrizes["preco_sem_ajuste"])
retorno_mensal_novo = preco_ajustado_mensal / preco_ajustado_mensal.shift(1) - 1

print(preco_ajustado_mensal.shape, preco_sem_ajuste_mensal.shape)
print("preços ajustados antes:", int(preco_mensal.notna().sum().sum()), "| depois:", int(preco_ajustado_mensal.notna().sum().sum()))
sem_ajuste_antigo = matrizes["preco_sem_ajuste"].resample("ME").last()
print("preços sem ajuste antes:", int(sem_ajuste_antigo.notna().sum().sum()), "| depois:", int(preco_sem_ajuste_mensal.notna().sum().sum()))
print("retornos válidos antes:", int(retorno_mensal.notna().sum().sum()), "| depois:", int(retorno_mensal_novo.notna().sum().sum()))
print(preco_ajustado_mensal.notna().sum(axis=1).head(3))
print(preco_ajustado_mensal.notna().sum(axis=1).tail(3))

A regra apagou 2.067 preços (44.978 para 42.911) e os retornos válidos caíram de 43.333 para 40.756.

In [ ]:
apagados = preco_mensal.notna() & preco_ajustado_mensal.isna()
print(int(apagados.sum().sum()), "preços apagados")
print(apagados.sum().nlargest(8))

## 7. Liquidez de 12 meses

Mediana, nos últimos 12 meses (mínimo 6), da mediana mensal do volume diário, ignorando dias sem negociação. Piso: R$ 1 milhão por dia (1000 em R$ mil).

In [ ]:
LIQUIDEZ_MINIMA = 1000.0
DIA_SEM_NEGOCIACAO = "zeros"


def mediana_movel_12m(volume_diario):
    volume_por_mes = volume_diario.groupby(volume_diario.index.to_period("M")).median()
    mediana_movel = volume_por_mes.rolling(12, min_periods=6).median()
    mediana_movel.index = mediana_movel.index.to_timestamp(how="end").normalize()
    return mediana_movel


def volume_com_dias_zerados(volume_diario, preco_ajustado_diario):
    tem_preco = preco_ajustado_diario.notna()
    vida = tem_preco.cummax() & tem_preco.iloc[::-1].cummax().iloc[::-1]
    return volume_diario.fillna(0).where(vida)


liquidez_ignorando_dias = mediana_movel_12m(matrizes["volume"])
liquidez_com_zeros = mediana_movel_12m(volume_com_dias_zerados(matrizes["volume"], matrizes["preco_ajustado"]))
liquidez_12m = liquidez_com_zeros if DIA_SEM_NEGOCIACAO == "zeros" else liquidez_ignorando_dias

In [ ]:
print(liquidez_12m.shape)
print(liquidez_12m["PETR4"].dropna().tail(3))
print(liquidez_12m["SOND5"].dropna().tail(3))
passa_no_piso = liquidez_12m >= LIQUIDEZ_MINIMA
print(passa_no_piso.sum(axis=1).iloc[[0, 5, 12, 40, 80, -1]])

Dia sem negociação conta como zero dentro da vida da ação. O parâmetro DIA_SEM_NEGOCIACAO permite voltar à regra do legado ("ignorar").


## 8. Balanços alinhados ao mês

Um balanço só é público depois de um prazo: 45 dias após o fim do trimestre (ITR) e 90 dias após o 4T (DFP). Em cada fim de mês, a ação usa o último trimestre já publicado.

In [ ]:
import numpy as np


def disponivel_em(fim_do_trimestre):
    prazo = np.where(fim_do_trimestre.month == 12, 90, 45)
    return fim_do_trimestre + pd.to_timedelta(prazo, unit="D")


def contabil_disponivel_em(tabela_trimestral, datas_de_decisao):
    datas_de_decisao = pd.DatetimeIndex(datas_de_decisao)
    divulgacao = disponivel_em(tabela_trimestral.index)
    posicao = np.searchsorted(divulgacao.to_numpy(), datas_de_decisao.to_numpy(), side="right") - 1
    valores = tabela_trimestral.to_numpy(dtype=float)[np.clip(posicao, 0, None)]
    valores[posicao < 0] = np.nan
    return pd.DataFrame(valores, index=datas_de_decisao, columns=tabela_trimestral.columns)


datas_mensais = preco_ajustado_mensal.index
contab_pl = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)

In [ ]:
print(matrizes["patrimonio_liquido"]["PETR4"].loc["2017-03-31":"2018-06-30"])
print(contab_pl["PETR4"].loc["2017-03-31":"2018-06-30"])

In [ ]:
contab_pl = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)
contab_divida_liquida = contabil_disponivel_em(matrizes["divida_liquida"], datas_mensais)
contab_ebitda = contabil_disponivel_em(matrizes["ebitda"], datas_mensais)
contab_da = contabil_disponivel_em(matrizes["depreciacao"], datas_mensais)

for nome, tabela in [("pl", contab_pl), ("divida_liquida", contab_divida_liquida), ("ebitda", contab_ebitda), ("da", contab_da)]:
    print(nome, tabela.shape, int(tabela.notna().sum().sum()))

In [ ]:
tabela_teste = pd.DataFrame({"X": [1.0, 2.0, 3.0]}, index=pd.to_datetime(["2020-09-30", "2020-12-31", "2021-03-31"]))
datas_teste = pd.to_datetime(["2020-11-10", "2020-11-14", "2021-03-30", "2021-03-31", "2021-05-14", "2021-05-16"])

resultado = contabil_disponivel_em(tabela_teste, datas_teste)["X"].tolist()
assert np.allclose(resultado, [np.nan, 1, 1, 2, 2, 3], equal_nan=True), resultado

alterada = tabela_teste.copy()
alterada.loc["2021-03-31", "X"] = 99.0
antes = contabil_disponivel_em(tabela_teste, datas_teste)
depois = contabil_disponivel_em(alterada, datas_teste)
assert antes.loc[:"2021-05-14"].equals(depois.loc[:"2021-05-14"])
print("defasagem ok")

In [ ]:
alterada = matrizes["patrimonio_liquido"].copy()
alterada.loc["2017-12-31"] = alterada.loc["2017-12-31"] * 2
antes = contabil_disponivel_em(matrizes["patrimonio_liquido"], datas_mensais)
depois = contabil_disponivel_em(alterada, datas_mensais)
assert antes.loc[:"2018-02-28"].equals(depois.loc[:"2018-02-28"])
assert not antes.loc["2018-03-31"].equals(depois.loc["2018-03-31"])
print("defasagem na tabela real ok")

Em cada fim de mês, cada ação usa o último balanço já publicado (45 dias após o 1T a 3T, 90 dias após o 4T), sem preencher com o valor anterior. O teste confirma o prazo e que alterar um trimestre não muda nenhuma data anterior à publicação dele.


## 9. Cadastro: o que existe antes de filtrar

Antes de excluir ações, vemos os tipos de ativo, segmentos, setores e classes que o cadastro traz.

In [ ]:
print(cadastro["tipo_de_ativo"].value_counts())
print(cadastro["segmento"].value_counts())
print(cadastro["setor_economico"].value_counts())
print(cadastro["classe"].value_counts())

In [ ]:
SETORES_EXCLUIDOS = ["Financeiro"]
SEGMENTOS_EXCLUIDOS = ["Balcão Organizado", "Bovespa Mais", "Bovespa Mais Nível 2", "BDR nível 3", "BDR nível 2", "BDR nível 1"]


def codigos_permitidos(cadastro):
    permitido = ~cadastro["setor_economico"].isin(SETORES_EXCLUIDOS)
    permitido &= ~cadastro["segmento"].isin(SEGMENTOS_EXCLUIDOS)
    permitido &= ~cadastro["classe"].astype(str).str.contains("Resg")
    return set(cadastro.loc[permitido, "codigo"])


permitidos = codigos_permitidos(cadastro)

print("excluídos por setor:", int(cadastro["setor_economico"].isin(SETORES_EXCLUIDOS).sum()))
print("excluídos por segmento:", int(cadastro["segmento"].isin(SEGMENTOS_EXCLUIDOS).sum()))
print("excluídos por classe resgatável:", int(cadastro["classe"].astype(str).str.contains("Resg").sum()))
print("permitidos:", len(permitidos), "de", cadastro["codigo"].nunique())
print("units entre os permitidos:", int(cadastro.loc[cadastro["codigo"].isin(permitidos), "classe"].astype(str).str.contains("UNT|UnN1").sum()))

Todos os 1.419 ativos são do tipo "Ação"; o tipo não filtra. Saem o setor Financeiro (162), os segmentos fora de bolsa (81) e as classes resgatáveis (4), e sobram 1.191. Pendências: ativos sem setor (não dá para saber se são financeiros) e as 27 units que seguem no universo.

## 10. Uma classe por empresa

A empresa é identificada pela raiz do ISIN (caracteres 3 a 6); sem ISIN, pelas quatro primeiras letras do código. A classe que fica é a de maior liquidez entre as que passam nos outros filtros.

In [ ]:
def raiz_da_empresa(cadastro):
    isin = cadastro["isin"].astype(str).str.strip()
    raiz = isin.str[2:6].where(isin.str.len() >= 6, cadastro["codigo"].astype(str).str[:4])
    return pd.Series(raiz.values, index=cadastro["codigo"].values)


sem_isin_valido = cadastro["isin"].astype(str).str.strip().str.len() < 6
raiz = raiz_da_empresa(cadastro)
raiz_permitidos = raiz[list(permitidos)]

print(raiz[["PETR3", "PETR4", "VALE3", "ABEV3"]])
print("sem ISIN válido:", int(sem_isin_valido.sum()), "| exemplos:", cadastro.loc[sem_isin_valido, "isin"].unique()[:3])
print("empresas entre os permitidos:", raiz_permitidos.nunique(), "| códigos:", len(raiz_permitidos))
print(raiz_permitidos.value_counts().head(5))

752 empresas entre os 1.191 códigos permitidos. Ativos sem ISIN (507, todos cancelados) são agrupados pelas quatro primeiras letras do código.

## 11. Universo elegível por mês

Elegível = permitida no cadastro, com preço no mês, preço sem ajuste de pelo menos R$ 1, liquidez no piso, valor de mercado positivo e, por empresa, só a classe de maior liquidez.

In [ ]:
PRECO_MINIMO = 1.0


def elegivel_por_data(fechamento, preco_sem_ajuste_mensal, liquidez, valor_mercado, cadastro):
    colunas = [codigo for codigo in fechamento.columns if codigo in permitidos]
    raiz_colunas = raiz_da_empresa(cadastro)[colunas]
    passa = fechamento[colunas].notna()
    passa &= preco_sem_ajuste_mensal[colunas] >= PRECO_MINIMO
    passa &= liquidez[colunas] >= LIQUIDEZ_MINIMA
    passa &= valor_mercado[colunas] > 0
    liquidez_valida = liquidez[colunas].where(passa)
    elegivel = pd.DataFrame(False, index=fechamento.index, columns=fechamento.columns)
    for data in fechamento.index:
        linha = liquidez_valida.loc[data].dropna()
        if linha.empty:
            continue
        tabela = pd.DataFrame({"liquidez": linha, "raiz": raiz_colunas[linha.index]})
        escolhidos = tabela.sort_values("liquidez", ascending=False).drop_duplicates("raiz").index
        elegivel.loc[data, escolhidos] = True
    return elegivel


elegivel = elegivel_por_data(preco_ajustado_mensal, preco_sem_ajuste_mensal, liquidez_12m, matrizes["valor_mercado"], cadastro)
print(elegivel.shape)
print(elegivel.sum(axis=1).iloc[[0, 5, 12, 40, 80, -1]])
print("média por mês:", round(elegivel.sum(axis=1).iloc[5:].mean(), 1))

In [ ]:
financeiras = cadastro.loc[cadastro["setor_economico"] == "Financeiro", "codigo"]
print("financeiras elegíveis:", int(elegivel.reindex(columns=financeiras).sum().sum()))
print("PDGR3 em out/2025:", bool(elegivel.loc["2025-10-31", "PDGR3"]))
por_empresa = elegivel.T.groupby(raiz.reindex(elegivel.columns).values).sum().max().max()
print("máximo de classes por empresa em um mês:", int(por_empresa))

Média de 153,8 elegíveis por mês (111 em ago/2017, 148 em dez/2025). Conferido: nenhuma financeira, no máximo uma classe por empresa e PDGR3 fora em out/2025.

## 12. Saídas e possíveis migrações

Lista as ações que param de ter preço antes do fim da base, com o retorno do último mês. Também procura códigos da mesma empresa que começam perto da saída (possível migração).

In [ ]:
def tabela_de_saidas(fechamento, ultimo_mes_dos_dados):
    ultimo_mes_com_preco = pd.to_datetime(fechamento.apply(lambda coluna: coluna.last_valid_index()))
    saiu = ultimo_mes_com_preco < ultimo_mes_dos_dados
    retorno = fechamento.pct_change(fill_method=None)
    saidas = pd.DataFrame({"ultimo_mes_com_preco": ultimo_mes_com_preco[saiu]})
    saidas["retorno_no_ultimo_mes"] = [retorno.at[mes, codigo] for codigo, mes in saidas["ultimo_mes_com_preco"].items()]
    return saidas


saidas = tabela_de_saidas(preco_ajustado_mensal, preco_ajustado_mensal.index[-2])

print(len(saidas), "ações com último preço antes de", preco_ajustado_mensal.index[-2].date())
print(saidas["retorno_no_ultimo_mes"].describe())
print(saidas.reindex(["JBSS3", "JBSS32"]))

In [ ]:
raiz = raiz_da_empresa(cadastro)
raiz = raiz[~raiz.index.duplicated()].reindex(preco_ajustado_mensal.columns)
quatro_letras = pd.Series([c[:4] for c in preco_ajustado_mensal.columns], index=preco_ajustado_mensal.columns)
primeiro_mes = pd.to_datetime(preco_ajustado_mensal.apply(lambda coluna: coluna.first_valid_index()))

candidatos = []
for codigo, mes_saida in saidas["ultimo_mes_com_preco"].items():
    mesma_empresa = (raiz == raiz[codigo]) | (quatro_letras == quatro_letras[codigo])
    comeca_perto = (primeiro_mes - mes_saida).abs() <= pd.Timedelta(days=62)
    for novo in preco_ajustado_mensal.columns[(mesma_empresa & comeca_perto).values]:
        if novo != codigo:
            candidatos.append({"saiu": codigo, "ultimo_mes": mes_saida.date(),
                               "novo": novo, "primeiro_mes_do_novo": primeiro_mes[novo].date()})

candidatos = pd.DataFrame(candidatos)
print(len(candidatos), "pares")
print(candidatos.head(40).to_string())

188 ações saem da base antes de nov/2025 (149 com retorno no último mês). Dos 9 pares de códigos da mesma empresa perto de uma saída, alguns parecem conversões de classe (JBSS3→JBSS32, SUZB5→SUZB3, CAMB4→CAMB3); o resto é início da base ou série de um mês. Os pares confirmados recebem tratamento no mês da troca (seção seguinte).

## 13. Retornos extremos

Conta os retornos acima de +100% e abaixo de -50% entre as ações elegíveis. Não removemos nada por tamanho: só listamos para investigar caso a caso.

In [ ]:
elegivel_no_mes_anterior = elegivel.shift(1, fill_value=False)
retorno_do_universo = retorno_mensal_novo.where(elegivel_no_mes_anterior)
r_universo = retorno_do_universo.stack().dropna()

print(len(r_universo), "retornos de ações elegíveis no mês anterior")
print((r_universo > 1).sum(), "acima de +100% |", (r_universo < -0.5).sum(), "abaixo de -50%")
print(r_universo.nlargest(10))
print(r_universo.nsmallest(10))

15.346 retornos de ações elegíveis: 10 acima de +100% e 40 abaixo de -50%. Vários parecem eventos reais (covid em mar/2020, Americanas em jan/2023); as altas em empresas em reestruturação precisam de conferência com o preço sem ajuste e a quantidade de ações.

In [ ]:
retorno_sem_ajuste = preco_sem_ajuste_mensal / preco_sem_ajuste_mensal.shift(1) - 1
variacao_acoes = (
    matrizes["acoes"]
    .reindex(index=preco_ajustado_mensal.index, columns=preco_ajustado_mensal.columns)
    .pct_change(fill_method=None)
)

extremos = r_universo[(r_universo > 1) | (r_universo < -0.5)].rename("ajustado").to_frame()
extremos["sem_ajuste"] = [retorno_sem_ajuste.at[d, c] for d, c in extremos.index]
extremos["var_acoes"] = [variacao_acoes.at[d, c] for d, c in extremos.index]

print(extremos.sort_values("ajustado").round(3).to_string())

Dos 50 extremos, 46 têm retorno ajustado igual ao sem ajuste (eventos reais, como covid e Americanas). Dois têm sinal oposto, RCSL3 (dez/2022) e GOLL54 (jun/2025, ações x2.860), e são candidatos a erro de ajuste. Falta contar casos assim na base inteira antes de definir uma regra geral.

In [ ]:
valor = matrizes["valor_mercado"].reindex(index=preco_ajustado_mensal.index, columns=preco_ajustado_mensal.columns)
var_valor_mercado = valor.pct_change(fill_method=None)

conferencia = pd.concat(
    {"ajustado": retorno_do_universo, "sem_ajuste": retorno_sem_ajuste,
     "var_acoes": variacao_acoes, "var_valor_mercado": var_valor_mercado},
    axis=1,
).stack(level=1).dropna(subset=["ajustado"])
conferencia["diferenca"] = conferencia["ajustado"] - conferencia["sem_ajuste"]

opostos = conferencia[(conferencia["ajustado"] * conferencia["sem_ajuste"] < 0) & (conferencia["diferenca"].abs() > 0.3)]

print(len(conferencia), "retornos conferidos")
print(len(opostos), "com sinal oposto e diferença acima de 30 pontos")
print(opostos.sort_values("diferenca").round(3).to_string())
print(conferencia[conferencia["ajustado"] > 1].round(3).to_string())

37 dos 42 casos de sinal oposto são desdobramentos e grupamentos bem ajustados (o ajustado acompanha o valor de mercado). As outras 8 altas acima de +100% têm os três retornos iguais: movimento real. Cinco casos não batem com nenhuma medida: GOLL54 (jun/2025), RCSL3 (dez/2022), BRAP4 (dez/2021), JSLG3 (dez/2025) e AZEV4 (fev/2025).

In [ ]:
LIMITE_DIVERGENCIA = 0.3

conferencia["dif_sem_ajuste"] = (conferencia["ajustado"] - conferencia["sem_ajuste"]).abs()
conferencia["dif_valor_mercado"] = (conferencia["ajustado"] - conferencia["var_valor_mercado"]).abs()

for limite in [0.2, 0.3, 0.5]:
    n = int(((conferencia["dif_sem_ajuste"] > limite) & (conferencia["dif_valor_mercado"] > limite)).sum())
    print("limite", limite, "->", n, "casos")

inexplicado = (conferencia["dif_sem_ajuste"] > LIMITE_DIVERGENCIA) & (conferencia["dif_valor_mercado"] > LIMITE_DIVERGENCIA)
print(conferencia[inexplicado].round(3).to_string())

Proposta, a decidir com o grupo: sinalizar como suspeito o retorno em que o ajustado diverge do sem ajuste e do valor de mercado por mais de 1,0. O limite foi escolhido depois de ver os dados; qualquer valor entre ~0,73 e ~1,26 dá os mesmos 2 casos (RCSL3 dez/2022 e GOLL54 jun/2025). Padrão: manter, com a lista guardada.

In [ ]:
LIMITE_DIVERGENCIA = 1.0
AJUSTE_INEXPLICADO = "manter"

dif_sem_ajuste = (retorno_mensal_novo - retorno_sem_ajuste).abs()
dif_valor_mercado = (retorno_mensal_novo - var_valor_mercado).abs()
suspeitos = (dif_sem_ajuste > LIMITE_DIVERGENCIA) & (dif_valor_mercado > LIMITE_DIVERGENCIA)

casos = suspeitos.stack()
casos = casos[casos].index
print(len(casos), "retornos suspeitos na base inteira")
for data, codigo in casos:
    print(data.date(), codigo, round(retorno_mensal_novo.at[data, codigo], 3),
          round(retorno_sem_ajuste.at[data, codigo], 3), round(var_valor_mercado.at[data, codigo], 3))

if AJUSTE_INEXPLICADO == "anular":
    retorno_mensal_tratado = retorno_mensal_novo.mask(suspeitos)
else:
    retorno_mensal_tratado = retorno_mensal_novo.copy()

print(int(retorno_mensal_novo.notna().sum().sum()), "->", int(retorno_mensal_tratado.notna().sum().sum()), "retornos válidos")

10 retornos suspeitos na base inteira (2 no universo elegível). "Suspeito" = o ajustado diverge do sem ajuste e do valor de mercado por mais de 1,0; ainda não é erro confirmado. Padrão "manter": nada foi alterado.

In [ ]:
def olhar_caso(codigo, data):
    inicio = data.replace(day=1)
    tabela = pd.concat(
        [matrizes["preco_ajustado"][codigo], matrizes["preco_sem_ajuste"][codigo]],
        axis=1, keys=["ajustado", "sem_ajuste"],
    ).loc[inicio - pd.Timedelta(days=10): data].dropna(how="all")
    tabela["ret_ajustado"] = tabela["ajustado"].pct_change(fill_method=None)
    tabela["ret_sem_ajuste"] = tabela["sem_ajuste"].pct_change(fill_method=None)
    dia = tabela.loc[inicio:, "ret_ajustado"].abs().idxmax()
    return tabela.loc[dia - pd.Timedelta(days=5): dia + pd.Timedelta(days=5)].round(3)


for data, codigo in casos:
    print(codigo, data.date())
    print(olhar_caso(codigo, data).to_string())
    print()

Dos 10 suspeitos, 6 mostram o ajustado saltando num dia em que o preço cai (GSHP3, RCSL3 dez/2022, AZEV3, BRPR3, FASA3, GOLL54): provável erro de ajuste. RCSL3 nov/2018 e RVEE3 são ambíguos. IDVL4 e PDGR3 são falsos positivos da regra mensal (ajuste correto). Nada foi alterado; a lista vai ao grupo.

In [ ]:
pares = [("JBSS3", "JBSS32"), ("VVAR3", "BHIA3"), ("LAME4", "AMER3"),
         ("SUZB5", "SUZB3"), ("CAMB4", "CAMB3"), ("AFLU3", "AFLU5"), ("BMGB11", "BMGB4")]
sem_ajuste_diario = matrizes["preco_sem_ajuste"]

linhas = []
for antigo, novo in pares:
    if antigo not in sem_ajuste_diario.columns or novo not in sem_ajuste_diario.columns:
        linhas.append({"antigo": antigo, "novo": novo})
        continue
    ultimo = sem_ajuste_diario[antigo].last_valid_index()
    primeiro = sem_ajuste_diario[novo].first_valid_index()
    p_antigo = sem_ajuste_diario[antigo].loc[ultimo]
    p_novo = sem_ajuste_diario[novo].loc[primeiro]
    linhas.append({"antigo": antigo, "novo": novo, "ultimo_dia_antigo": ultimo.date(),
                   "primeiro_dia_novo": primeiro.date(), "dias_entre": (primeiro - ultimo).days,
                   "preco_antigo": p_antigo, "preco_novo": p_novo, "razao": round(p_novo / p_antigo, 3)})

print(pd.DataFrame(linhas).to_string())

In [ ]:
setor = (
    cadastro.assign(codigo=cadastro["codigo"].astype(str).str.strip())
    .drop_duplicates("codigo").set_index("codigo")["setor_economico"]
    .reindex(elegivel.columns)
)
sem_setor = (setor.isna() | setor.astype(str).str.strip().isin(["-", "", "nan"])).values
print(int(sem_setor.sum()), "ações sem setor no cadastro")
print("elegíveis sem setor por mês (média):", round(elegivel.loc[:, sem_setor].sum(axis=1).mean(), 1))
print(sorted(elegivel.columns[sem_setor & elegivel.any().values])[:40])

In [ ]:
print([c for c in matrizes["preco_sem_ajuste"].columns if c.startswith(("VVAR", "VIIA", "BHIA", "LAME", "AMER", "BTOW", "B2W"))])

## 14. Mês da troca de código

Em cinco trocas confirmadas (JBSS3, SUZB5, CAMB4, AFLU3, LAME4), o retorno do código antigo no mês da troca é: retorno do antigo até o último pregão dele, vezes o retorno do novo desde o primeiro pregão seguinte. VVAR→BHIA3 não precisa (o BHIA3 já traz o histórico). Parâmetro TRATAR_MIGRACAO.

In [ ]:
PARES_MIGRACAO = [("JBSS3", "JBSS32"), ("SUZB5", "SUZB3"), ("CAMB4", "CAMB3"),
                  ("AFLU3", "AFLU5"), ("LAME4", "AMER3")]
TRATAR_MIGRACAO = True


def retorno_do_mes_da_troca(antigo, novo, diario, mensal):
    serie_antiga = diario[antigo].dropna()
    ultimo_dia = serie_antiga.index[-1]
    ultimo_mes = mensal[antigo].last_valid_index()
    mes_da_troca = mensal.index[mensal.index.get_loc(ultimo_mes) + 1]
    primeiro_dia_novo = diario[novo].loc[ultimo_dia + pd.Timedelta(days=1):].first_valid_index()
    parte_antiga = serie_antiga.iloc[-1] / mensal.at[ultimo_mes, antigo]
    parte_nova = mensal.at[mes_da_troca, novo] / diario[novo].loc[primeiro_dia_novo]
    return mes_da_troca, parte_antiga * parte_nova - 1


retorno_mensal_final = retorno_mensal_novo.copy()
if TRATAR_MIGRACAO:
    for antigo, novo in PARES_MIGRACAO:
        mes, retorno = retorno_do_mes_da_troca(antigo, novo, matrizes["preco_ajustado"], preco_ajustado_mensal)
        print(antigo, "->", novo, mes.date(), "antes:", retorno_mensal_final.at[mes, antigo], "| depois:", round(retorno, 3))
        retorno_mensal_final.at[mes, antigo] = retorno

Quatro das cinco trocas deram retorno plausível no mês da troca (JBSS3 -2,7%, SUZB5 -10,6%, CAMB4 +9,4%, LAME4 -8,8%). AFLU3 ficou vazio: o código novo não tem preço de fim de mês. Com TRATAR_MIGRACAO = False, nada muda.

## 15. Gravar os arquivos do 01

Grava em data/processed os arquivos do contrato (seção 26 do documento), que o 02, 03 e 04 leem. Os .pkl não vão para o Git: cada pessoa roda este caderno.

In [ ]:
PASTA_SAIDA = Path("../data/processed")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

colunas = preco_ajustado_mensal.columns
datas = preco_ajustado_mensal.index

cadastro_universo = (
    cadastro.assign(codigo=cadastro["codigo"].astype(str).str.strip(),
                    raiz_isin=raiz_da_empresa(cadastro).values)
    .drop_duplicates("codigo").set_index("codigo")
    [["isin", "classe", "tipo_de_ativo", "setor_economico", "subsetor", "segmento", "situacao", "raiz_isin"]]
    .reindex(colunas)
)

arquivos_saida = {
    "cadastro_universo": cadastro_universo,
    "retorno_mensal": retorno_mensal_final,
    "preco_sem_ajuste_mensal": preco_sem_ajuste_mensal,
    "valor_mercado_mensal": matrizes["valor_mercado"].reindex(index=datas, columns=colunas),
    "liquidez_12m": liquidez_12m,
    "contab_pl": contab_pl,
    "contab_divida_liquida": contab_divida_liquida,
    "contab_ebitda": contab_ebitda,
    "contab_da": contab_da,
    "elegivel": elegivel,
    "saidas": saidas,
    "cdi_mensal": cdi_mensal.reindex(datas),
    "ibov_mensal": ibov_ret.reindex(datas),
}

for nome, tabela in arquivos_saida.items():
    if isinstance(tabela, pd.DataFrame) and tabela.shape == (len(datas), len(colunas)):
        arquivos_saida[nome] = tabela.reindex(index=datas, columns=colunas)

for nome, objeto in arquivos_saida.items():
    pd.to_pickle(objeto, PASTA_SAIDA / f"{nome}.pkl")
    print(nome, objeto.shape)

## 16. DI de 1 ano e IPCA
Alinha o DI (B3) e o IPCA (IBGE) às 106 datas e grava em `data/processed`. O DI só entra no diagnóstico de juros; o IPCA, no retorno real opcional.

In [ ]:
di = pd.read_csv(PASTA_RAW / '18_di_1a_mensal.csv', parse_dates=['mes', 'data_pregao']).set_index('mes')
ipca = pd.read_csv(PASTA_RAW / '19_ipca_mensal.csv', parse_dates=['mes']).set_index('mes')

assert di.index.equals(datas) and ipca.index.equals(datas)
atraso = (di.index - di['data_pregao']).dt.days
assert atraso.between(0, 5).all()
assert di['taxa_252'].notna().all() and ipca['ipca_decimal'].notna().all()

di_mensal = di[['taxa_252', 'delta_taxa']]
ipca_mensal = ipca['ipca_decimal']
pd.to_pickle(di_mensal, PASTA_SAIDA / 'di_mensal.pkl')
pd.to_pickle(ipca_mensal, PASTA_SAIDA / 'ipca_mensal.pkl')
print(di_mensal.shape, ipca_mensal.shape, int(atraso.max()), 'dias de atraso máximo do pregão')